# Mapping Intertidal Elevation with Google Earth Engine and the FES2022 Tide Model

### A reproducible, end-to-end tutorial — from a blank environment to a validated elevation surface

This notebook builds an **intertidal Digital Elevation Model (DEM)** of a coastal flat
entirely from open data: the Landsat surface-reflectance archive and a global ocean-tide
model. It is written for graduate researchers who may be new to Python *or* to remote
sensing — every concept is introduced before it is used, but the treatment is kept
quantitatively honest rather than simplified to the point of being wrong.

#### The physical idea

The intertidal zone is, by definition, the part of the seabed that is exposed at low
water and submerged at high water. A satellite in a sun-synchronous orbit passes over at
an essentially fixed *solar* time but an effectively **random tidal phase**: across many
years of acquisitions, a given pixel is imaged across the full range of water levels.

For each pixel we therefore observe a binary state — *wet* or *dry* — as a function of
the instantaneous tide height. A pixel that is dry only at the lowest tides sits low in
the tidal frame; one that is dry across almost all tides sits high. The **tide height at
which a pixel transitions from dry to wet is, to first order, the elevation of the
sediment surface at that location.** This is the *waterline* method (Mason et al., 1995;
Bishop-Taylor et al., 2019); the implementation here follows the logic of the Digital
Earth Australia (DEA) Intertidal product (Bishop-Taylor et al., 2019, *Estuarine, Coastal
and Shelf Science*).

#### Why Google Earth Engine (GEE)?

GEE is a planetary-scale geospatial analysis platform that co-locates a petabyte-scale
satellite archive with compute. We never download imagery; we send a *recipe* of
operations to Google's servers and retrieve only the result. This makes a multi-decadal,
multi-sensor analysis tractable on a laptop.

#### Why an external tide model?

GEE has no knowledge of sea level at acquisition time. We obtain it from **FES2022**
(Finite Element Solution 2022; Lyard et al., 2021), a global barotropic tide model that
reconstructs water level from harmonic constituents. We evaluate it locally with
`eo-tides`/`pyTMD` and attach the result to each scene.

---

#### What you will need
- A Google account and a (free, non-commercial) Earth Engine **Cloud Project**
- **Python ≥ 3.12** (required by the pinned versions of `geemap` and `rioxarray`)
- ~2–4 GB of disk for the FES2022 constituent files
- An independent elevation reference (e.g. a lidar DTM) *if* you wish to validate — optional but strongly recommended

#### Estimated run time
Setup (Stage 0–2) is the slow part. Once authenticated, a small area (a few km across)
runs in minutes because the heavy computation is server-side.

> **How to read this notebook.** Markdown cells state the *why*; the code cell immediately
> below performs the *what*. Read top to bottom on the first pass — later stages depend on
> objects created earlier.


## Stage 0 — Software environment

> **Do the installation before launching this notebook.** Install the environment from
> the terminal with **`uv`** (a fast, modern Python project manager), *then* start Jupyter
> from inside it. Full cross-platform instructions — including a conda fallback — are in the
> accompanying **`../README.md`**. In brief, from the project folder:
>
> ```bash
> uv sync --frozen          # installs Python 3.12 + the exact locked package set
> uv run jupyter lab        # then open this notebook
> ```
>
> You do not need to install GDAL yourself — it ships inside the `rasterio`/`pyogrio`
> wheels. This Stage 0 therefore **verifies** the environment rather than installing into
> it. If a check below fails, return to `../README.md`.

We rely on the following packages. Their roles:

| Package | Role in this workflow |
|---|---|
| `earthengine-api` | The Python client for Google Earth Engine |
| `geemap` | Interactive mapping and convenience I/O between GEE and the local environment |
| `eo-tides` | High-level tide-height modelling from global tide models |
| `pyTMD` | The numerical engine `eo-tides` calls to evaluate tidal constituents (installed *via* `eo-tides`) |
| `geopandas` | Vector geometry (areas of interest, polygons) |
| `xarray`, `rioxarray` | Labelled n-dimensional arrays and raster I/O |
| `numpy`, `pandas`, `matplotlib` | Numerics, tabular data, static figures |

#### Why versions are pinned (and why `pyTMD` is not installed directly)
A tutorial is only reproducible if everyone resolves the same dependency tree. The project
files (`pyproject.toml` + `uv.lock`) pin each package to a verified, mutually compatible
release (verified June 2026) and target **Python ≥ 3.12** (required by `geemap` and
`rioxarray`). The `uv.lock` file records the full resolved tree of all 197 packages, so
`uv sync --frozen` reproduces the identical environment on any machine. Note that `pyTMD`
is *not* installed explicitly: `eo-tides` requires `pyTMD<3.0.0` and resolves the correct
version (2.2.9.1) itself — installing it manually risks a conflict.


In [ ]:
# --- Verify the environment is correctly set up (no installation happens here) ---
import sys

# 1) Python version: the pinned geemap/rioxarray require >= 3.12.
assert sys.version_info >= (3, 12), (
    f"Python >= 3.12 required (you have {sys.version.split()[0]}). "
    f"See ../README.md to create the 'intertidal' environment."
)
print(f"Python {sys.version.split()[0]} — OK.")

# 2) All required packages importable?
import importlib
required = ["ee", "geemap", "eo_tides", "pyTMD",
            "geopandas", "rioxarray", "xarray", "numpy", "pandas", "matplotlib"]
missing = []
for mod in required:
    try:
        importlib.import_module(mod)
    except ImportError:
        missing.append(mod)
if missing:
    raise SystemExit(
        f"Missing packages: {missing}. Your Jupyter kernel may be pointing at the "
        f"wrong environment. See the 'wrong kernel' note in ../README.md."
    )
print("All required packages import correctly.")

In [ ]:
# --- Report the resolved versions (paste this into your methods / supplementary info) ---
import importlib.metadata as im
for pkg in ["earthengine-api", "geemap", "eo-tides", "pyTMD",
            "geopandas", "rioxarray", "xarray", "numpy", "pandas", "matplotlib"]:
    try:
        print(f"  {pkg:<18} {im.version(pkg)}")
    except im.PackageNotFoundError:
        print(f"  {pkg:<18} NOT INSTALLED")

## Stage 1 — Authenticating with Earth Engine

Earth Engine access is free for research and education but requires a one-time
registration tied to a Google Cloud Project.

1. Visit **https://earthengine.google.com/** and register.
2. Create or select a Cloud Project (a free *non-commercial* project is sufficient).
3. Note the **project ID** (e.g. `ee-yourname`); you will supply it below.

The cell below initialises the client. On first use it triggers an interactive
authentication flow (a browser window in which you grant access and paste back a token).
This is stored locally, so subsequent sessions on the same machine skip it.


In [ ]:
import ee
import geemap

EE_PROJECT = "ee-replace-me"   # <-- EDIT: your Cloud Project ID

try:
    ee.Initialize(project=EE_PROJECT)
    print("Earth Engine initialised (existing credentials found).")
except Exception:
    ee.Authenticate()                  # interactive, one time per machine
    ee.Initialize(project=EE_PROJECT)
    print("Authentication successful; Earth Engine initialised.")

In [ ]:
# Connectivity check: query the size of a small Landsat-8 subset.
_probe = ee.ImageCollection("LANDSAT/LC08/C02/T1_L2").limit(5)
print("Connection OK. Probe collection size:", _probe.size().getInfo())

## Stage 2 — Obtaining and configuring the FES2022 tide model

### What the model is

A barotropic ocean-tide model represents sea-surface height as a sum of **harmonic
constituents** — sinusoids at the astronomical forcing frequencies (the principal
lunar semidiurnal M2, principal solar S2, lunisolar diurnal K1, and so on). FES2022
provides the amplitude and phase of each constituent on a global grid; evaluating the
sum at a given location and time reconstructs the predicted tide height. Because the
forcing is astronomical, the model extrapolates backwards in time as readily as forwards
— a property we rely on if we later analyse historical Landsat-5 imagery.

### Obtaining the files

FES2022 is distributed (free, with registration) through **AVISO+**:

1. Complete the data-access registration at
   **https://www.aviso.altimetry.fr/en/data/data-access/registration-form.html**,
   requesting the **FES (Finite Element Solution) tide** product.
2. After approval, download the **`fes2022b`** *ocean tide* constituent files
   (one NetCDF per constituent: `m2_fes2022.nc`, `s2_fes2022.nc`, `k1_fes2022.nc`, …).

Arrange them in the directory layout `pyTMD` expects:

```
TIDE_DIR/
└── fes2022b/
    └── ocean_tide/
        ├── m2_fes2022.nc
        ├── s2_fes2022.nc
        ├── k1_fes2022.nc
        └── ...
```

> **Why this step is manual.** AVISO requires authenticated download behind a registration
> form, which cannot be automated from the notebook. It is a one-time acquisition.

The cell below records the path and verifies the constituent files are discoverable.


In [ ]:
import os
from pathlib import Path

TIDE_DIR = "./tide_models"      # <-- EDIT: directory containing fes2022b/ocean_tide/...
TIDE_MODEL = "FES2022"

os.environ["EO_TIDES_TIDE_MODELS"] = TIDE_DIR   # eo-tides reads this

ocean_tide = Path(TIDE_DIR) / "fes2022b" / "ocean_tide"
if ocean_tide.exists():
    constituents = sorted(ocean_tide.glob("*.nc"))
    print(f"Found {len(constituents)} constituent files in {ocean_tide}")
    for f in constituents[:5]:
        print("  -", f.name)
    if len(constituents) > 5:
        print(f"  ... ({len(constituents) - 5} more)")
    if len(constituents) < 30:
        print("\nNote: a complete FES2022 set has ~34 constituents. "
              "A partial set degrades prediction accuracy.")
else:
    print("Directory not found:", ocean_tide)
    print("Download FES2022 (see text above) and set TIDE_DIR accordingly.")

## Stage 3 — Defining the study area

The analysis is bounded by an **area of interest (AOI)**, here a rectangle in geographic
coordinates (WGS84 / EPSG:4326), specified by its western, southern, eastern, and northern
limits in decimal degrees.

#### Choosing coordinates
Read corner coordinates from any web map (e.g. https://geojson.io) or supply your own.
Begin with a **small** area (a few kilometres across): the waterline method needs many
clear observations spanning the tidal range, and a compact AOI keeps both the scene count
per tidal band and the runtime manageable while you learn the workflow.

#### A caution on AOI extent and tides
The single-point tide evaluation used later (Stage 7) assumes the tide is approximately
uniform across the AOI. This holds for a small flat but **fails for a large or elongated
estuary**, where the tidal wave propagates and both phase and range vary spatially. We
return to this limitation explicitly in Stage 7 and the closing discussion.


In [ ]:
# === EDIT to your study area ===
WEST, SOUTH, EAST, NORTH = 3.96, 51.55, 4.10, 51.62   # example: part of the Oosterschelde, NL
SITE_NAME = "study_site"

aoi  = ee.Geometry.Rectangle([WEST, SOUTH, EAST, NORTH])
bbox = (WEST, SOUTH, EAST, NORTH)
mid_lon = (WEST + EAST) / 2.0
mid_lat = (SOUTH + NORTH) / 2.0

print(f"Site: {SITE_NAME}")
print(f"  bounds  : W={WEST}, S={SOUTH}, E={EAST}, N={NORTH}")
print(f"  centroid: {mid_lat:.4f} N, {mid_lon:.4f} E")

In [ ]:
# Visual check. Confirm the red rectangle covers the intended intertidal area.
m = geemap.Map(center=[mid_lat, mid_lon], zoom=11)
m.add_basemap("HYBRID")
m.addLayer(aoi, {"color": "red"}, "AOI")
m

## Stage 4 — Temporal window and key parameters

The method's precision scales with how densely the observed water levels sample the tidal
range. A longer observation window yields more clear scenes and a more even coverage of
tide heights, improving the per-pixel transition estimate — at the cost of assuming the
morphology is approximately stationary over that window. Choosing the window is therefore
a substantive scientific decision, not a default.

Sensor choice: we use **Landsat Collection 2, Level 2** (atmospherically corrected surface
reflectance). The Landsat archive extends to 1984, enabling historical analysis with a
consistent ~30 m sensor — useful if you later compare two epochs (e.g. to detect
accretion/erosion).

Parameters exposed here:
- `START`, `END` — observation window.
- `MAX_CLOUD` — scene-level cloud-cover ceiling (per the metadata field), a coarse
  pre-filter applied *before* the per-pixel cloud mask.
- `WATER_THRESHOLD` — the NDWI value above which a pixel is classified *wet* (Stage 6).


In [ ]:
START = "2021-01-01"
END   = "2024-12-31"
MAX_CLOUD = 80          # %; scene-level pre-filter
WATER_THRESHOLD = 0.0   # NDWI > threshold => wet. 0.0 is a common starting value.

print(f"Window     : {START} to {END}")
print(f"Cloud ceil.: {MAX_CLOUD}%")
print(f"NDWI wet th: {WATER_THRESHOLD}")

## Stage 5 — Building a cloud-masked, multi-sensor Landsat collection

This stage assembles the imagery server-side. Nothing is computed yet — GEE constructs a
lazy description of the operations and defers execution until a result is requested.

#### What we do and why

1. **Merge Landsat 5, 7, 8 and 9.** Combining missions maximises the number of
   observations and, for historical windows, brings in Landsat-5. The sensors differ in
   band *numbering*, so we harmonise to common names (`green`, `red`, `nir`).
2. **Scene-level cloud pre-filter** on the `CLOUD_COVER` metadata property.
3. **Per-pixel cloud and shadow masking** using the `QA_PIXEL` quality band. Collection 2
   encodes pixel quality as bit flags; **bit 3 = cloud**, **bit 4 = cloud shadow**. We
   retain only pixels where both flags are clear. (Masking per pixel is essential: a scene
   passing the 80% ceiling can still contain local cloud over the AOI.)
4. **Scaling to physical surface reflectance.** Collection 2 Level 2 stores reflectance as
   scaled integers; the official conversion is `reflectance = DN x 0.0000275 - 0.2`.

> A subtlety we accept for clarity: Landsat 5/7 (TM/ETM+) and 8/9 (OLI) have slightly
> different spectral response in the green and NIR bands. For a binary wet/dry decision via
> NDWI this cross-sensor difference is minor, but it is a known source of bias in
> reflectance-continuity studies and worth noting if you extend this work.


In [ ]:
def mask_and_scale(img):
    \"\"\"Apply the QA_PIXEL cloud/shadow mask and convert to surface reflectance.\"\"\"
    qa = img.select("QA_PIXEL")
    cloud_or_shadow = qa.bitwiseAnd(1 << 3).Or(qa.bitwiseAnd(1 << 4))  # bit3=cloud, bit4=shadow
    clear = cloud_or_shadow.eq(0)

    optical = img.select("SR_B.").multiply(0.0000275).add(-0.2)        # scaled int -> reflectance
    return (optical
            .updateMask(clear)
            .copyProperties(img, ["system:time_start", "CLOUD_COVER"]))


def harmonize_bandnames(img, sensor):
    \"\"\"Rename mission-specific band numbers to common names: green, red, nir.\"\"\"
    if sensor in ("LT05", "LE07"):           # TM / ETM+
        src = ["SR_B2", "SR_B3", "SR_B4", "QA_PIXEL"]
    else:                                     # OLI (L8/L9)
        src = ["SR_B3", "SR_B4", "SR_B5", "QA_PIXEL"]
    return img.select(src, ["green", "red", "nir", "QA_PIXEL"])


In [ ]:
def build_landsat_collection(aoi, start, end, max_cloud):
    \"\"\"Return a merged, cloud-masked, band-harmonised Landsat collection.\"\"\"
    sensors = {
        "LT05": "LANDSAT/LT05/C02/T1_L2",
        "LE07": "LANDSAT/LE07/C02/T1_L2",
        "LC08": "LANDSAT/LC08/C02/T1_L2",
        "LC09": "LANDSAT/LC09/C02/T1_L2",
    }
    parts = []
    for sensor, cid in sensors.items():
        col = (ee.ImageCollection(cid)
               .filterBounds(aoi)
               .filterDate(start, end)
               .filter(ee.Filter.lt("CLOUD_COVER", max_cloud))
               .map(mask_and_scale)
               .map(lambda im, s=sensor: harmonize_bandnames(im, s)))
        parts.append(col)

    merged = parts[0]
    for c in parts[1:]:
        merged = merged.merge(c)
    return merged.sort("system:time_start")


landsat = build_landsat_collection(aoi, START, END, MAX_CLOUD)
n_scenes = landsat.size().getInfo()
print(f"Clear-enough Landsat scenes in window: {n_scenes}")
if n_scenes < 20:
    print("WARNING: few scenes. The per-pixel transition estimate will be noisy. "
          "Extend the window or relax MAX_CLOUD.")

## Stage 6 — The water index (NDWI)

To label each pixel *wet* or *dry* we use the **Normalized Difference Water Index**
(McFeeters, 1996):

$$\text{NDWI} = \frac{\rho_{\text{green}} - \rho_{\text{NIR}}}{\rho_{\text{green}} + \rho_{\text{NIR}}}$$

The index exploits the strong absorption of near-infrared radiation by liquid water:
open water yields high NDWI, while exposed sediment and vegetation yield low values. A
threshold (`WATER_THRESHOLD`) converts the continuous index to a binary wet/dry decision.

> **Caveats worth stating at PhD level.** A fixed global threshold is a simplification.
> Wet sediment, thin water films, suspended-sediment-laden water, and sun glint all shift
> NDWI and can bias the classification near the waterline — precisely the region we care
> about. More robust pipelines use a per-scene or Otsu-derived adaptive threshold and/or
> the MNDWI variant (green–SWIR; Xu, 2006), which suppresses built-up and soil signals.
> We use a fixed NDWI here for transparency; treat the threshold as a parameter to test
> in sensitivity analysis.


In [ ]:
def add_ndwi(img):
    ndwi = img.normalizedDifference(["green", "nir"]).rename("ndwi")
    return img.addBands(ndwi)

landsat_ndwi = landsat.map(add_ndwi)

# Diagnostic: temporal-mean NDWI. Persistent water is dark blue, persistently dry is brown;
# intermediate (intertidal) pixels take intermediate values.
mean_ndwi = landsat_ndwi.select("ndwi").mean().clip(aoi)
m2 = geemap.Map(center=[mid_lat, mid_lon], zoom=11)
m2.addLayer(mean_ndwi, {"min": -0.5, "max": 0.5,
                        "palette": ["saddlebrown", "white", "navy"]}, "Mean NDWI")
m2.addLayer(aoi, {"color": "red"}, "AOI")
m2

## Stage 7 — Modelling tide height at each acquisition

We now connect each satellite observation to a water level. For every scene we take the
acquisition timestamp and evaluate FES2022 at the AOI to obtain the predicted tide height.

The computation is performed **locally** (the model files are on disk), so we extract only
the *timestamps* from Earth Engine.

#### A note on the reference datum
`eo-tides`/FES2022 returns tide height relative to **mean sea level** (the model's zero).
The elevations we ultimately produce inherit this datum. If you need heights relative to a
chart or national datum (e.g. NAP, LAT, MSL of a specific epoch), apply the appropriate
offset — and be explicit about the datum in any reported result.

#### Spatial sampling of the tide
We evaluate the tide at the **AOI centroid only**. This is defensible for a small flat
where tidal phase and range are near-uniform. For a large or elongated system, replace the
single point with a grid of points and interpolate a per-pixel tide field; the single-point
assumption is the dominant geometric error source otherwise.


In [ ]:
import pandas as pd
import numpy as np

# Acquisition times (epoch milliseconds) from Earth Engine
times_ms = landsat_ndwi.aggregate_array("system:time_start").getInfo()
scene_times = pd.to_datetime(times_ms, unit="ms", utc=True)
print(f"Retrieved {len(scene_times)} acquisition timestamps.")
print("Earliest:", scene_times.min(), "| Latest:", scene_times.max())

In [ ]:
from eo_tides.model import model_tides

# Predict tide height (m, relative to model MSL) at the AOI centroid for every timestamp.
tide_df = model_tides(
    x=[mid_lon],
    y=[mid_lat],
    time=scene_times,
    model=TIDE_MODEL,
    directory=TIDE_DIR,
)

# Reduce to a time-indexed series of tide heights.
tide_by_time = tide_df.reset_index().groupby("time")["tide_height"].mean()

print(f"Tidal range sampled by the scenes: "
      f"{tide_by_time.min():.2f} m to {tide_by_time.max():.2f} m "
      f"(span {tide_by_time.max() - tide_by_time.min():.2f} m)")
print("\nFirst few predictions:")
print(tide_by_time.head())

In [ ]:
# Diagnostic: how evenly do the observations sample the tidal range?
# Uneven sampling (e.g. a sun-synchronous bias toward certain tide phases) directly
# limits which elevations can be resolved.
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(7, 3.2))
ax.hist(tide_by_time.values, bins=20, color="#3b528b", edgecolor="white")
ax.set_xlabel("Modelled tide height at acquisition (m, MSL)")
ax.set_ylabel("Number of scenes")
ax.set_title("Tidal sampling of the observation stack")
fig.tight_layout()
plt.show()

## Stage 8 — Attaching tide height to each scene in Earth Engine

The tide heights computed locally must be associated with the corresponding server-side
images so that subsequent reductions can group scenes by water level. We attach the tide
height as an image property (`tide`), preserving acquisition order.

> **Implementation caveat.** We map a locally computed list onto the collection by index.
> This assumes the order of `aggregate_array("system:time_start")` matches the order of the
> collection as iterated — which holds here because both derive from the same sorted
> collection. Scenes for which the tide lookup failed are dropped. For a production pipeline,
> a join keyed on the timestamp value is more robust than an index map.


In [ ]:
# Map timestamp (ms) -> tide height, then push as a per-image property, in order.
ms_to_tide = {int(t.timestamp() * 1000): float(h) for t, h in tide_by_time.items()}
tide_values = ee.List([ms_to_tide.get(int(t), -999) for t in times_ms])

img_list = landsat_ndwi.toList(landsat_ndwi.size())

def _set_tide(i):
    i = ee.Number(i)
    return ee.Image(img_list.get(i)).set("tide", tide_values.get(i))

landsat_tide = ee.ImageCollection(
    ee.List.sequence(0, landsat_ndwi.size().subtract(1)).map(_set_tide)
).filter(ee.Filter.neq("tide", -999))   # drop failed lookups

print("Scenes carrying a valid tide property:", landsat_tide.size().getInfo())

## Stage 9 — Estimating elevation from the waterline transition

This is the core inference. The reasoning, made precise:

For a pixel at true elevation $z$, the probability of being observed *wet* increases
monotonically with tide height $h$: it is dry when $h < z$ and wet when $h > z$, blurred by
classification noise and sub-pixel topography into a smooth transition. The tide height at
which the **wet probability crosses 0.5** is therefore an estimator of $z$.

We approximate this non-parametrically:

1. **Partition** the observed tidal range into `N_BINS` equal bands.
2. **Composite** each band into the per-pixel *wet fraction* — the mean of the binary
   wet masks for scenes whose tide falls in that band.
3. **Locate the transition**: per pixel, take the lowest band centre at which the wet
   fraction reaches 0.5. That tide height is the elevation estimate.

This piecewise estimator is a deliberately transparent stand-in for the smoother
interval-interpolation used in the DEA Intertidal product (which also propagates a
per-pixel uncertainty). It exposes the mechanism clearly; see the closing discussion for
the accuracy implications.

> **Resolution vs. support trade-off.** More bins give finer elevation resolution but fewer
> scenes per band, increasing the variance of each wet-fraction estimate. With `n_scenes`
> on the order of a few dozen, ~15–20 bins is a reasonable balance. This is the key tunable.


In [ ]:
N_BINS = 20

t_min = float(tide_by_time.min())
t_max = float(tide_by_time.max())
edges = np.linspace(t_min, t_max, N_BINS + 1)
centers = (edges[:-1] + edges[1:]) / 2.0

def wet_fraction_band(lo, hi, center):
    \"\"\"Per-pixel wet fraction for scenes with tide in [lo, hi); carries band centre.\"\"\"
    in_band = landsat_tide.filter(ee.Filter.And(
        ee.Filter.gte("tide", lo), ee.Filter.lt("tide", hi)))
    wet = in_band.map(lambda im: im.select("ndwi").gt(WATER_THRESHOLD)
                                  .rename("wet").toFloat())
    frac = wet.mean().rename("wet_frac")                          # mean of binary = fraction
    return frac.addBands(ee.Image.constant(center).rename("tide_center").toFloat())

bands = [wet_fraction_band(float(edges[i]), float(edges[i+1]), float(centers[i]))
         for i in range(N_BINS)]
bands_col = ee.ImageCollection(bands)
print(f"Constructed {N_BINS} tidal bands spanning {t_min:.2f}–{t_max:.2f} m.")

In [ ]:
# Per pixel: the lowest band centre at which wet_fraction >= 0.5 is the transition tide,
# i.e. the elevation estimate. Implemented as the minimum tide_center over bands where the
# pixel is (on balance) wet.
def transition_candidate(img):
    is_wet = img.select("wet_frac").gte(0.5)
    return img.select("tide_center").updateMask(is_wet).rename("elev_candidate")

elevation = (bands_col.map(transition_candidate)
                      .min()                 # lowest tide at which pixel turns wet
                      .rename("elevation")
                      .clip(aoi))
print("Elevation surface computed (units: m relative to model MSL).")

## Stage 10 — Visual inspection

We display the elevation surface on an interactive map. Lower elevations (submerged across
most tides) render dark; higher elevations (exposed across most tides) render light. Inspect
for plausibility: channels should be low, flats and shoals high, and the spatial pattern
should be coherent rather than speckled. Heavy speckle indicates too few scenes per band or
a poorly chosen NDWI threshold.


In [ ]:
vis = {"min": t_min, "max": t_max,
       "palette": ["#440154", "#3b528b", "#21918c", "#5ec962", "#fde725"]}  # viridis

m3 = geemap.Map(center=[mid_lat, mid_lon], zoom=12)
m3.add_basemap("HYBRID")
m3.addLayer(elevation, vis, "Intertidal elevation (m)")
m3.addLayer(aoi, {"color": "red"}, "AOI")
m3.add_colorbar(vis, label="Elevation (m, MSL)")
m3

## Stage 11 — Exporting the result

Two export routes, chosen by AOI size:

- **A — direct to the local filesystem** (`geemap.ee_export_image`): convenient for small
  areas, subject to a synchronous size limit (~32 MB / a few million pixels).
- **B — to Google Drive** (`ee.batch.Export`): an asynchronous server-side task suitable for
  large areas; the GeoTIFF appears in Drive when the task completes.

We export at 30 m (Landsat native) in EPSG:4326. For analysis you may prefer a local
projected CRS (e.g. a UTM zone) to work in metres; set `crs` accordingly.


In [ ]:
# --- A. Direct download (small AOI) ---
out_tif = f"{SITE_NAME}_elevation.tif"
try:
    geemap.ee_export_image(elevation, filename=out_tif, scale=30,
                           region=aoi, file_per_band=False)
    print("Exported to:", out_tif)
except Exception as e:
    print("Direct export failed (AOI likely too large). Use the Drive export below.")
    print("Reason:", e)

In [ ]:
# --- B. Export to Google Drive (large AOI) ---
# Uncomment to run; monitor at https://code.earthengine.google.com/tasks
# task = ee.batch.Export.image.toDrive(
#     image=elevation,
#     description=f"{SITE_NAME}_elevation",
#     folder="EE_exports",
#     fileNamePrefix=f"{SITE_NAME}_elevation",
#     region=aoi,
#     scale=30,
#     crs="EPSG:4326",
#     maxPixels=1e10,
# )
# task.start()
# print("Export task started.")

## Stage 12 — Static figure for reporting

A publication-style static map, retrieved as a NumPy array via `geemap.ee_to_numpy`.


In [ ]:
arr = geemap.ee_to_numpy(elevation, region=aoi, default_value=np.nan)
if arr is not None:
    arr = np.squeeze(arr).astype(float)
    fig, ax = plt.subplots(figsize=(8, 6))
    im = ax.imshow(arr, cmap="viridis", vmin=t_min, vmax=t_max)
    ax.set_title(f"Intertidal elevation — {SITE_NAME}")
    ax.set_xticks([]); ax.set_yticks([])
    cbar = fig.colorbar(im, ax=ax, shrink=0.78)
    cbar.set_label("Elevation (m, MSL)")
    fig.tight_layout()
    fig.savefig(f"{SITE_NAME}_elevation.png", dpi=150, bbox_inches="tight")
    plt.show()
    print("Figure saved:", f"{SITE_NAME}_elevation.png")
else:
    print("ee_to_numpy returned no array (AOI may be too large for synchronous retrieval).")

## Stage 13 — Validation (recommended)

No elevation product should be reported without an accuracy assessment. If you have an
independent reference DEM (e.g. an airborne lidar DTM), validate against it. The standard
procedure:

1. Reproject both surfaces to a common grid and CRS, and **reconcile the vertical datum**
   (the satellite-derived heights are relative to model MSL; the reference may be NAP/LAT/
   ellipsoidal). A datum mismatch appears as a constant bias.
2. Restrict the comparison to genuinely intertidal pixels (exclude permanent water and
   permanently dry land, where the method does not estimate elevation).
3. Report, at minimum, **bias (mean error)**, **MAE**, and **RMSE**, ideally stratified by
   elevation band — error is typically largest near the extremes of the sampled tidal range,
   where few observations constrain the transition.

The skeleton below shows the comparison logic for a co-registered reference array
`ref_arr` aligned to `arr` from Stage 12. Adapt the loading/alignment to your reference
data (e.g. with `rioxarray` and `rio.reproject_match`).


In [ ]:
# Template — supply your own co-registered reference DEM as `ref_arr` (NumPy, same grid as `arr`).
def validate(estimate, reference):
    \"\"\"Return bias, MAE, RMSE over pixels where both surfaces are finite.\"\"\"
    mask = np.isfinite(estimate) & np.isfinite(reference)
    if mask.sum() == 0:
        return None
    diff = estimate[mask] - reference[mask]      # positive => estimate higher than reference
    return {
        "n":    int(mask.sum()),
        "bias": float(np.mean(diff)),
        "MAE":  float(np.mean(np.abs(diff))),
        "RMSE": float(np.sqrt(np.mean(diff**2))),
    }

# Example (requires ref_arr defined and aligned to `arr`):
# stats = validate(arr, ref_arr)
# print(stats)
print("Provide a co-registered reference DEM as `ref_arr`, then call validate(arr, ref_arr).")

## Discussion and limitations

You now have a complete, reproducible workflow from open imagery and a tide model to an
intertidal elevation surface. Before applying it in research, internalise its assumptions.

**Method simplifications relative to operational products.** The 0.5-crossing estimator over
discrete tidal bands is intentionally transparent. Operational pipelines (DEA Intertidal;
Bishop-Taylor et al., 2019) fit a smoother relationship across the full observation set and
propagate a **per-pixel uncertainty**, achieving mean absolute errors on the order of
decimetres. The estimator here will be coarser and provides no uncertainty layer; treat its
output as instructional unless you extend it.

**Tidal sampling and the sun-synchronous bias.** Landsat's fixed overpass time means certain
tidal phases are systematically over- or under-sampled at a given site. Elevations near the
extremes of the *sampled* range (not the true tidal range) are poorly constrained. The Stage 7
histogram is your diagnostic; sparse tails there translate directly into unreliable elevations.

**Single-point tide.** Acceptable for a compact flat, inadequate for an extended estuary with
spatially varying phase and range. Generalise to a tidal grid for large systems.

**Classification near the waterline.** A fixed NDWI threshold is most error-prone exactly where
the method is most sensitive — at the land–water boundary — due to wet sediment, thin films,
turbidity and glint. Test threshold sensitivity; consider MNDWI or an adaptive threshold.

**Datum and morphological stationarity.** Heights are relative to model MSL and assume the
morphology is stationary over the observation window. State the datum explicitly, and keep the
window short enough that stationarity is defensible for your site — or split into epochs and
analyse change directly.

#### Suggested extensions
- Two-epoch differencing (e.g. 1985–1987 vs. 2023–2025) to quantify accretion/erosion.
- A per-pixel uncertainty layer from the spread of the wet-fraction transition.
- Sensitivity analyses over `WATER_THRESHOLD` and `N_BINS`.
- Per-pixel tide modelling for large AOIs.

#### Key references
- Mason, D.C. et al. (1995). Construction of an inter-tidal digital elevation model by the 'water-line' method. *Geophysical Research Letters*.
- McFeeters, S.K. (1996). The use of the Normalized Difference Water Index (NDWI). *International Journal of Remote Sensing*.
- Xu, H. (2006). Modification of NDWI to enhance open water features (MNDWI). *International Journal of Remote Sensing*.
- Bishop-Taylor, R. et al. (2019). Between the tides: modelling the elevation of Australia's exposed intertidal zone. *Estuarine, Coastal and Shelf Science*.
- Lyard, F.H. et al. (2021). FES2014/FES2022 global ocean tide atlas. *Ocean Science*.
